In [2]:
pip install langchain

In [3]:
import sys
print(sys.version)


3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [4]:
pip install cassio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 9.6 MB/s eta 0:00:00


In [5]:
pip install openai

In [6]:
!pip install h5py
!pip install typing-extensions
!pip install wheel
'''
h5py	-> Handles large dataset storage (HDF5 format)
typing-extensions	-> Provides advanced Python typing support
wheel ->	Enables faster and reliable package installation'''

In [27]:
pip install tiktoken
'''Purpose:

Token counting

Optimizes chunk size

Why used:
Improves performance and avoids token limit errors.'''

In [8]:
pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 33.4 MB/s eta 0:00:00


In [9]:
!pip install datasets

In [12]:
!pip install langchain-community
from langchain_community.vectorstores import Cassandra

#support for dataset retrieval with hugging Face
from datasets import load_dataset

import cassio

In [ ]:
!pip install PyPDF2

In [14]:
!pip install PyPDF2
#package to read pdfs
from PyPDF2 import PdfReader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.5 MB/s eta 0:00:00


In [21]:
ASTRA_DB_APPLICATION_TOKEN = "**"
ASTRA_DB_ID = "**"

OPENAI_API_KEY = "**"

In [16]:
pdfreader = PdfReader('/content/ncert_tb.pdf')

In [28]:
# @title
from typing_extensions import Concatenate
raw_text = ''
for i,page in enumerate(pdfreader.pages):
  content = page.extract_text()
  if content:
    raw_text += content

# to read the data in each page and extract that data and stored in raw data from each page

In [18]:
raw_text

'166 BIOLOGY\nYou have alr eady studied the or ganisation of a flowering plant in Chapter\n5. Have you ever thought about where and how the structures like roots,\nstems, leaves, flowers, fruits and seeds arise and that too in an orderly\nsequence? Y ou ar e, by now, awar e of the ter ms seed, seedling, plantlet,\nmatur e plant. Y ou have also seen that tr ees continue to incr ease in height\nor girth over a period of time. However , the leaves, flowers and fruits of the\nsame tree not only have limited dimensions but also appear and fall\nperiodically and some time repeatedly. Why does vegetative phase precede\nflowering in a plant? All plant organs are made up of a variety of tissues; is\nthere any relationship between the structure of a cell, a tissue, an organ\nand the function they perform? Can the structure and the function of these\nbe altered? All cells of a plant are descendents of the zygote. The question\nis, then, why and how do they have different structural and functional

In [23]:
#To initialize connection to database in lagchain
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN,database_id=ASTRA_DB_ID)

In [24]:
pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.3 MB/s eta 0:00:00


In [25]:
#create langchain embeddings and langchain objects used later
from langchain_openai import OpenAI
from langchain_openai import OpenAIEmbeddings
llm = OpenAI(openai_api_key=OPENAI_API_KEY)
embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

In [26]:
#creating vector database in langchain using astra_db
astra_vector_store = Cassandra(
    embedding=embedding,
    table_name="qa_mini_demo",
    session=None,
    keyspace=None,
)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
#this done the task of creating text chunks of a definite size
from langchain.text_splitter import CharacterTextSplitter
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 800,
    chunk_overlap = 200,
    length_function = len,
)
texts = text_splitter.split_text(raw_text)

In [ ]:
texts[:50]

In [ ]:
# this create text embeddings as well as store those in db
astra_vector_store.add_texts(texts[:50])
print("Inserted %i headlines." % len(texts[:50]))
astra_vector_index = VectorStoreIndexWrapper(vectorstore=astra_vector_store)

In [ ]:
first_question = True
while True:
  if first_question:
    query_text = input("Enter your question or type quit to exit").strip()
  else:
    query_text = input("What's your next question or type quit to exit").strip()
  if(query_text.lower() == "quit"):
    print("Thank you and feel free to ask questions according to budget")
    break
  if query_text == "":
    continue
  first_question = False
  print("\nQUESTION: \"%s\"" %query_text)
  answer = astra_vector_index.query(query_text,llm=llm).strip()
  print("Answer: \"%s\"\n" %answer)
  print("FIRST DOCUMENTS BY RELEVENCE: ")
  for doc, score in astra_vector_store.similarity_search_with_score(query_text,k=4):
    print("   [%0.4f] \"%s....\"" %(score,doc.page_content[:84]))